# Hands-on 03 · DataFrames y Spark SQL
**Curso:** Base de Datos Avanzadas y Big Data · **Semana 14** · **Unidad III**

**Caso:** control de calidad y análisis de pedidos de una distribuidora ficticia.

**Objetivo:** leer dos fuentes CSV, depurar pedidos, relacionar clientes y pedidos, calcular indicadores mediante DataFrames y SQL, validar y guardar resultados en Parquet.

Los datos son **simulados**. Ejecuta las celdas de arriba hacia abajo. Primero resuelve el caso guiado y luego el desafío.

## 0. Comprobar el entorno

Si tuviste el error *Python worker failed to connect back*, reinicia el kernel. La configuración debe ejecutarse **antes** de crear la SparkSession. No desactives el firewall.

In [1]:
import os, sys
from pathlib import Path
import pyspark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
print("Python:",sys.executable)
print("PySpark:",pyspark.__version__)
print("Carpeta de trabajo:",Path.cwd())

Python: C:\Users\UPN\.conda\envs\etl_bi\python.exe
PySpark: 4.2.0
Carpeta de trabajo: D:\EGCC\UPN_2026_2_BDA_26066\Semana14_22NOV\S14Lab


## 1. Crear SparkSession y probar el motor

Esta prueba trabaja con un DataFrame y no requiere archivos. Si falla, revisa el entorno antes de continuar. `local[2]` ejecuta Spark en una sola máquina.

In [2]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("UPN_HandsOn03_SparkSQL")
         .master("local[2]")
         .config("spark.driver.bindAddress","127.0.0.1")
         .config("spark.driver.host","127.0.0.1")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Versión de Spark:",spark.version)
print("Prueba:",spark.range(3).count())
assert spark.range(3).count() == 3

Versión de Spark: 4.2.0
Prueba: 3


## 2. Localizar las fuentes

Coloca `pedidos.csv` y `clientes.csv` en la misma carpeta que este notebook. Evitamos rutas absolutas de Windows.

In [5]:
from pathlib import Path
base = Path.cwd() # Directorio de trabajo actual
archivo_pedidos = base / "pedidos.csv"
archivo_clientes = base / "clientes.csv"
for p in [archivo_pedidos,archivo_clientes]:
    if not p.is_file():
        raise FileNotFoundError(f"No se encuentra {p.name}. Copia el CSV junto al notebook.")
    print("Encontrado:",p.name, "tamaño(bytes):",p.stat().st_size)

Encontrado: pedidos.csv tamaño(bytes): 611546
Encontrado: clientes.csv tamaño(bytes): 1808


## 3. Leer CSV con esquema explícito

Se utiliza un esquema definido para evitar inferencias incorrectas. El importe se lee como número decimal: los vacíos se convierten en valores nulos.

In [6]:
from pyspark.sql.types import StructType,StructField,StringType,DecimalType
from pyspark.sql import functions as F
esquema_pedidos = StructType([
 StructField("id_pedido",StringType(),True),
 StructField("fecha",StringType(),True),
 StructField("id_cliente",StringType(),True),
 StructField("categoria",StringType(),True),
 StructField("importe",DecimalType(12,2),True),
 StructField("estado",StringType(),True)
])
esquema_clientes = StructType([
 StructField("id_cliente",StringType(),True),
 StructField("region",StringType(),True),
 StructField("segmento",StringType(),True)
])
pedidos = spark.read.option("header",True).schema(esquema_pedidos).csv(str(archivo_pedidos))
clientes = spark.read.option("header",True).schema(esquema_clientes).csv(str(archivo_clientes))

clientes.printSchema()
clientes.show(5,truncate=False)

pedidos.printSchema()
pedidos.show(5,truncate=False)
print("Pedidos:",pedidos.count(),"Clientes:",clientes.count())

assert pedidos.count()==12024
assert clientes.count()==80

root
 |-- id_cliente: string (nullable = true)
 |-- region: string (nullable = true)
 |-- segmento: string (nullable = true)

+----------+-------+---------+
|id_cliente|region |segmento |
+----------+-------+---------+
|C001      |Norte  |Minorista|
|C002      |Centro |Minorista|
|C003      |Sur    |Mayorista|
|C004      |Oriente|Mayorista|
|C005      |Norte  |Mayorista|
+----------+-------+---------+
only showing top 5 rows
root
 |-- id_pedido: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- importe: decimal(12,2) (nullable = true)
 |-- estado: string (nullable = true)

+---------+----------+----------+----------+-------+---------+
|id_pedido|fecha     |id_cliente|categoria |importe|estado   |
+---------+----------+----------+----------+-------+---------+
|P000001  |2026-01-14|C012      |Hogar     |52.01  |Pendiente|
|P000002  |2026-01-27|C023      |Tecnología|79.02  |Cancelado|
|P000

## 4. Diagnosticar la calidad de datos

Calculamos los registros con importe faltante o no positivo, los IDs repetidos y los clientes sin correspondencia. **Todavía no eliminamos filas**.

Lo que se esta haciendo es una auditoria a los datos.

In [7]:
nulos = pedidos.filter(F.col("importe").isNull()).count()
no_positivos = pedidos.filter(F.col("importe") <= 0).count()
duplicados = pedidos.count() - pedidos.dropDuplicates().count()
huerfanos = (pedidos.select("id_cliente").distinct()
            .join(clientes.select("id_cliente"),"id_cliente","left_anti").count())
print("Nulos (filas, incluye duplicados):", nulos)
print("No positivos (filas):",no_positivos)
print("Duplicados exactos:",duplicados)
print("Clientes desconocidos diferentes:",huerfanos)

resultado = (nulos, no_positivos, duplicados, huerfanos)
esperado = (90, 40, 24, 30)
assert resultado == esperado, f"Esperado {esperado}, obtenido {resultado}"


Nulos (filas, incluye duplicados): 90
No positivos (filas): 40
Duplicados exactos: 24
Clientes desconocidos diferentes: 30


## 5. Definir una política y limpiar

Política del caso: quitar duplicados exactos, excluir importe faltante/no positivo y pedidos cuyo cliente no exista. No confundas esta regla didáctica con una regla universal.

Se realiza la limpieza de los datos.

In [8]:
pedidos_unicos = pedidos.dropDuplicates()
pedidos_correctos = pedidos_unicos.filter(F.col("importe").isNotNull() & (F.col("importe")>0))
pedidos_validos = pedidos_correctos.join(clientes,"id_cliente","inner")
print("Pedidos únicos:",pedidos_unicos.count())
print("Pedidos válidos:",pedidos_validos.count())
pedidos_validos.show(5, truncate=False)

## Validacion
assert pedidos_unicos.count()==12000
assert pedidos_validos.count()==11840

Pedidos únicos: 12000
Pedidos válidos: 11840
+----------+---------+----------+---------+-------+---------+-------+---------+
|id_cliente|id_pedido|fecha     |categoria|importe|estado   |region |segmento |
+----------+---------+----------+---------+-------+---------+-------+---------+
|C068      |P000057  |2026-01-18|Hogar    |177.57 |Entregado|Oriente|Minorista|
|C036      |P000585  |2026-01-04|Hogar    |568.85 |Entregado|Oriente|Minorista|
|C029      |P000708  |2026-06-04|Alimentos|423.58 |Entregado|Norte  |Mayorista|
|C077      |P000756  |2026-02-24|Alimentos|333.06 |Entregado|Norte  |Mayorista|
|C050      |P000819  |2026-05-30|Higiene  |647.69 |Entregado|Centro |Minorista|
+----------+---------+----------+---------+-------+---------+-------+---------+
only showing top 5 rows


## 6. Construir indicadores con DataFrames

Calculamos cantidad e importe total por región. La tabla combina datos procedentes de dos fuentes distintas.

In [9]:
indicadores_df = (pedidos_validos.groupBy("region")
  .agg(F.count("id_pedido").alias("cantidad_pedidos"),
       F.sum("importe").alias("importe_total"))
  .orderBy("region"))
indicadores_df.show(truncate=False)

# Validaciones

assert indicadores_df.agg(F.sum("cantidad_pedidos")).first()[0] == 11840

total_regiones = indicadores_df.agg(F.sum("importe_total")).first()[0]
total_directo = pedidos_validos.agg(F.sum("importe")).first()[0]
assert total_regiones == total_directo


+-------+----------------+-------------+
|region |cantidad_pedidos|importe_total|
+-------+----------------+-------------+
|Centro |2961            |1100123.95   |
|Norte  |2961            |1098961.06   |
|Oriente|2959            |1099322.05   |
|Sur    |2959            |1097411.64   |
+-------+----------------+-------------+



## 7. Obtener el mismo resultado con Spark SQL

Creamos una vista temporal que estará disponible durante la SparkSession y ejecutamos SQL. Debemos comprobar la equivalencia de ambos enfoques.

In [10]:
pedidos_validos.createOrReplaceTempView("pedidos_validos")
indicadores_sql = spark.sql("""
SELECT region,
       COUNT(*) AS cantidad_pedidos,
       SUM(importe) AS importe_total
FROM pedidos_validos
GROUP BY region
ORDER BY region
""")
indicadores_sql.show(truncate=False)
comparacion = (indicadores_df.exceptAll(indicadores_sql).count()
              + indicadores_sql.exceptAll(indicadores_df).count())
assert comparacion == 0
print("¿Son equivalentes los resultados?", comparacion == 0)

+-------+----------------+-------------+
|region |cantidad_pedidos|importe_total|
+-------+----------------+-------------+
|Centro |2961            |1100123.95   |
|Norte  |2961            |1098961.06   |
|Oriente|2959            |1099322.05   |
|Sur    |2959            |1097411.64   |
+-------+----------------+-------------+

¿Son equivalentes los resultados? True


In [11]:
# Se puede combinar

spark.sql("SELECT * FROM pedidos_validos WHERE importe > 500").groupBy("region").count().show()

+-------+-----+
| region|count|
+-------+-----+
|    Sur|  927|
|Oriente|  933|
|  Norte|  928|
| Centro|  934|
+-------+-----+



## 8. Estadísticas y verificación

Se comparan la suma total y el promedio sobre los pedidos válidos. Para cantidades monetarias mantenemos `DecimalType`.

In [12]:
estadisticas = pedidos_validos.agg(
 F.count("id_pedido").alias("n"),
 F.sum("importe").alias("total"),
 F.avg("importe").alias("promedio"),
 F.min("importe").alias("minimo"),
 F.max("importe").alias("maximo"))
estadisticas.show(truncate=False)
assert estadisticas.first()["n"] == 11840

+-----+----------+----------+------+------+
|n    |total     |promedio  |minimo|maximo|
+-----+----------+----------+------+------+
|11840|4395818.70|371.268471|25.00 |717.77|
+-----+----------+----------+------+------+



## 9. Guardar y volver a leer en Parquet

Spark crea una **carpeta** que contiene los archivos Parquet, no un único archivo. La guardamos localmente y comprobamos las filas recuperadas.

In [13]:
salida = base / "salida_indicadores_parquet"
indicadores_df.write.mode("overwrite").parquet(str(salida))
comprobacion = spark.read.parquet(str(salida))
comprobacion.orderBy("region").show()
assert comprobacion.count() == 4
print("Resultado guardado en:",salida)

Py4JJavaError: An error occurred while calling o187.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

## 10. Desafío autónomo (sin solución)
Una gerencia necesita conocer **la cantidad y el importe total de pedidos entregados por categoría**. Utiliza **solo los pedidos válidos** y entrega:

1. Una solución con la API de DataFrames.
2. Una solución equivalente con Spark SQL.
3. Una verificación que compruebe que ambos resultados coinciden.
4. Una explicación: ¿por qué no sería correcto contar pedidos cancelados como entregados?
5. Una consulta que identifique los dos segmentos de clientes con mayor importe total.

**Regla:** `estado == "Entregado"`. Explica cómo verificaste tus resultados y no te limites a copiar el código.

## Solución del desafío

In [14]:
entregados = pedidos_validos.filter(F.col("estado") == "Entregado")
por_categoria_df = (entregados.groupBy("categoria")
 .agg(F.count("id_pedido").alias("cantidad"), F.sum("importe").alias("total"))
 .orderBy("categoria"))
por_categoria_df.show(truncate=False)
entregados.createOrReplaceTempView("pedidos_entregados")
por_categoria_sql = spark.sql("""
 SELECT categoria, COUNT(*) AS cantidad, SUM(importe) AS total
 FROM pedidos_entregados GROUP BY categoria ORDER BY categoria
""")
assert por_categoria_df.exceptAll(por_categoria_sql).count() == 0
assert por_categoria_sql.exceptAll(por_categoria_df).count() == 0
segmentos = (pedidos_validos.groupBy("segmento")
 .agg(F.sum("importe").alias("importe_total"))
 .orderBy(F.desc("importe_total")))
segmentos.show(truncate=False)
assert segmentos.count() == 2

+----------+--------+---------+
|categoria |cantidad|total    |
+----------+--------+---------+
|Alimentos |987     |367491.76|
|Higiene   |988     |366894.80|
|Hogar     |985     |366220.47|
|Tecnología|986     |365070.90|
+----------+--------+---------+

+---------+-------------+
|segmento |importe_total|
+---------+-------------+
|Minorista|2254363.16   |
|Mayorista|2141455.54   |
+---------+-------------+



### Criterios de aceptación
- Se filtra el estado **Entregado** después de aplicar la política de calidad.
- No se confunden pedidos totales con pedidos entregados.
- DataFrames y SQL producen resultados equivalentes.
- La solución por segmento justifica correctamente el `join` y su cardinalidad.
- Se admiten soluciones alternativas verificables.

**Advertencia:** si falla `spark.range(3).count()` o una acción Python/RDD con `Python worker failed to connect back`, detener la práctica y diagnosticar el entorno; el archivo CSV no explica por sí mismo ese error.